# Resume Dataset Analysis & Skill Demand Visualization

This notebook provides an interactive exploration of the resume dataset, including data cleaning, skill extraction, and visualization of skill demands across different job categories.

## Table of Contents
1. [Data Loading and Exploration](#Data-Loading-and-Exploration)
2. [Data Preprocessing](#Data-Preprocessing)
3. [Skill Extraction](#Skill-Extraction)
4. [Exploratory Data Analysis](#Exploratory-Data-Analysis)
5. [Skill Demand Visualization](#Skill-Demand-Visualization)
6. [Insights and Conclusions](#Insights-and-Conclusions)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
from collections import Counter
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## Data Loading and Exploration

In [ ]:
# Load the resume dataset
# Note: Using a sample for faster processing with large datasets
SAMPLE_SIZE = 10000  # Adjust based on your system's capacity

try:
    df = pd.read_csv('../Dataset/Resume/Resume.csv', nrows=SAMPLE_SIZE)
    print(f"✅ Successfully loaded {len(df)} resumes")
    print(f"📊 Dataset shape: {df.shape}")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    df = None

In [ ]:
# Display basic information about the dataset
if df is not None:
    print("\n🔍 Dataset Information:")
    print(df.info())
    
    print("\n📋 First 5 rows:")
    display(df.head())
    
    print("\n📈 Basic Statistics:")
    print(f"Total resumes: {len(df)}")
    print(f"Unique categories: {df['Category'].nunique()}")
    print(f"Categories: {sorted(df['Category'].unique())}")

In [ ]:
# Analyze category distribution
if df is not None:
    plt.figure(figsize=(12, 6))
    
    category_counts = df['Category'].value_counts()
    
    plt.subplot(1, 2, 1)
    category_counts.head(10).plot(kind='bar', color='skyblue')
    plt.title('Top 10 Resume Categories')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Number of Resumes')
    
    plt.subplot(1, 2, 2)
    category_counts.head(10).plot(kind='pie', autopct='%1.1f%%', startangle=90)
    plt.title('Category Distribution (Top 10)')
    plt.ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Category Distribution:")
    for cat, count in category_counts.head(10).items():
        print(f"{cat}: {count} resumes ({count/len(df)*100:.1f}%)")

## Data Preprocessing

In [ ]:
# Define skill keywords for extraction
SKILL_KEYWORDS = {
    'technical': [
        'python', 'java', 'javascript', 'c++', 'c#', 'php', 'ruby', 'go', 'rust',
        'sql', 'mysql', 'postgresql', 'mongodb', 'redis', 'elasticsearch',
        'html', 'css', 'react', 'angular', 'vue', 'node.js', 'django', 'flask',
        'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'jenkins', 'git',
        'linux', 'windows', 'macos', 'tensorflow', 'pytorch', 'scikit-learn',
        'pandas', 'numpy', 'matplotlib', 'tableau', 'power bi'
    ],
    'soft': [
        'communication', 'leadership', 'teamwork', 'problem solving',
        'critical thinking', 'time management', 'adaptability', 'creativity',
        'emotional intelligence', 'conflict resolution', 'negotiation',
        'project management', 'agile', 'scrum', 'kanban'
    ],
    'business': [
        'marketing', 'sales', 'finance', 'accounting', 'strategy',
        'business development', 'customer service', 'operations',
        'supply chain', 'logistics', 'e-commerce', 'crm', 'erp'
    ],
    'industry': [
        'healthcare', 'finance', 'retail', 'manufacturing', 'education',
        'consulting', 'government', 'nonprofit', 'startup', 'enterprise'
    ]
}

def preprocess_text(text):
    """Clean and preprocess resume text."""
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def extract_skills(text):
    """Extract skills from resume text."""
    text = preprocess_text(text)
    found_skills = {'technical': [], 'soft': [], 'business': [], 'industry': []}
    
    for category, keywords in SKILL_KEYWORDS.items():
        for keyword in keywords:
            if keyword in text:
                found_skills[category].append(keyword)
    
    return found_skills

print("✅ Preprocessing functions defined")

## Skill Extraction

In [ ]:
# Extract skills from resumes (this may take some time)
if df is not None:
    print("🔄 Extracting skills from resumes...")
    
    skill_data = []
    for idx, row in df.iterrows():
        skills = extract_skills(row['Resume_str'])
        for skill_type, skill_list in skills.items():
            for skill in skill_list:
                skill_data.append({
                    'resume_id': row['ID'],
                    'category': row['Category'],
                    'skill': skill,
                    'skill_type': skill_type
                })
        
        # Progress indicator
        if idx % 1000 == 0:
            print(f"Processed {idx}/{len(df)} resumes...")
    
    skill_df = pd.DataFrame(skill_data)
    print(f"✅ Skill extraction complete! Found {len(skill_df)} skill mentions")
    print(f"📊 Unique skills: {skill_df['skill'].nunique()}")
else:
    skill_df = None

In [ ]:
# Display skill extraction results
if skill_df is not None and not skill_df.empty:
    print("\n📋 Skill Data Sample:")
    display(skill_df.head())
    
    print("\n📊 Skill Statistics:")
    print(f"Total skill mentions: {len(skill_df)}")
    print(f"Unique skills: {skill_df['skill'].nunique()}")
    print(f"Skill types: {skill_df['skill_type'].value_counts().to_dict()}")
    
    print("\n🏆 Top 10 Most Common Skills:")
    top_skills = skill_df['skill'].value_counts().head(10)
    for skill, count in top_skills.items():
        print(f"{skill}: {count} mentions")

## Exploratory Data Analysis

In [ ]:
# Analyze skills by category
if skill_df is not None and not skill_df.empty:
    plt.figure(figsize=(15, 10))
    
    # Skills by type
    plt.subplot(2, 2, 1)
    skill_type_counts = skill_df['skill_type'].value_counts()
    skill_type_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90)
    plt.title('Skill Types Distribution')
    plt.ylabel('')
    
    # Top skills overall
    plt.subplot(2, 2, 2)
    top_skills = skill_df['skill'].value_counts().head(15)
    top_skills.plot(kind='barh', color='lightcoral')
    plt.title('Top 15 Most Common Skills')
    plt.xlabel('Frequency')
    
    # Skills by category (top 5 categories)
    plt.subplot(2, 2, 3)
    top_categories = skill_df['category'].value_counts().head(5).index
    category_skill_data = skill_df[skill_df['category'].isin(top_categories)]
    pivot_data = pd.crosstab(category_skill_data['category'], category_skill_data['skill_type'])
    pivot_data.plot(kind='bar', stacked=True, ax=plt.gca())
    plt.title('Skill Types by Top Categories')
    plt.xticks(rotation=45, ha='right')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Average skills per resume by category
    plt.subplot(2, 2, 4)
    skills_per_resume = skill_df.groupby(['resume_id', 'category']).size().reset_index(name='skill_count')
    avg_skills_by_category = skills_per_resume.groupby('category')['skill_count'].mean().sort_values(ascending=False).head(10)
    avg_skills_by_category.plot(kind='bar', color='lightgreen')
    plt.title('Average Skills per Resume by Category')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Average Skills')
    
    plt.tight_layout()
    plt.show()

## Skill Demand Visualization

In [ ]:
# Create interactive visualizations
if skill_df is not None and not skill_df.empty:
    # Skills heatmap by category
    plt.figure(figsize=(12, 8))
    heatmap_data = pd.crosstab(skill_df['category'], skill_df['skill_type'])
    sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Skill Count'})
    plt.title('Skills Heatmap: Categories vs Skill Types')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Word cloud of all skills
    plt.figure(figsize=(12, 8))
    all_skills_text = ' '.join(skill_df['skill'].tolist())
    wordcloud = WordCloud(
        width=1200, height=800, 
        background_color='white',
        max_words=100, 
        colormap='viridis',
        random_state=42
    ).generate(all_skills_text)
    
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Skills Word Cloud', fontsize=16)
    plt.show()

In [ ]:
# Interactive Plotly visualizations
if skill_df is not None and not skill_df.empty:
    # Create subplots for interactive dashboard
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Top Skills by Category', 'Skill Types Distribution', 
                       'Skills per Category', 'Skill Demand Trends'),
        specs=[[{'type': 'bar'}, {'type': 'pie'}],
               [{'type': 'scatter'}, {'type': 'heatmap'}]]
    )
    
    # Top skills by category (stacked bar)
    top_categories = skill_df['category'].value_counts().head(5).index
    category_skill_counts = skill_df[skill_df['category'].isin(top_categories)]
    skill_category_pivot = pd.crosstab(category_skill_counts['category'], category_skill_counts['skill'])
    
    # Get top 8 skills overall
    top_8_skills = skill_df['skill'].value_counts().head(8).index
    
    for skill in top_8_skills:
        if skill in skill_category_pivot.columns:
            skill_data = skill_category_pivot[skill]
            fig.add_trace(
                go.Bar(name=skill, x=top_categories, y=skill_data),
                row=1, col=1
            )
    
    # Skill types pie chart
    skill_type_counts = skill_df['skill_type'].value_counts()
    fig.add_trace(
        go.Pie(labels=skill_type_counts.index, values=skill_type_counts.values,
              marker_colors=['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']),
        row=1, col=2
    )
    
    # Skills per category scatter plot
    skills_per_category = skill_df.groupby('category').size().reset_index(name='total_skills')
    unique_skills_per_category = skill_df.groupby('category')['skill'].nunique().reset_index(name='unique_skills')
    category_stats = pd.merge(skills_per_category, unique_skills_per_category, on='category')
    
    fig.add_trace(
        go.Scatter(
            x=category_stats['total_skills'], 
            y=category_stats['unique_skills'],
            mode='markers+text',
            text=category_stats['category'],
            textposition="top center",
            marker=dict(size=10, color='lightblue', line=dict(width=2, color='darkblue'))
        ),
        row=2, col=1
    )
    
    # Skill demand heatmap
    heatmap_data = pd.crosstab(skill_df['category'], skill_df['skill_type'])
    fig.add_trace(
        go.Heatmap(
            z=heatmap_data.values,
            x=heatmap_data.columns,
            y=heatmap_data.index,
            colorscale='Blues'
        ),
        row=2, col=2
    )
    
    fig.update_layout(
        height=800, 
        title_text="Interactive Resume Skills Analysis Dashboard",
        showlegend=True
    )
    fig.update_xaxes(title_text="Categories", row=1, col=1)
    fig.update_yaxes(title_text="Skill Count", row=1, col=1)
    fig.update_xaxes(title_text="Total Skills", row=2, col=1)
    fig.update_yaxes(title_text="Unique Skills", row=2, col=1)
    
    fig.show()

## Insights and Conclusions

In [ ]:
# Generate insights and summary
if skill_df is not None and df is not None:
    print("🎯 KEY INSIGHTS FROM RESUME ANALYSIS\n")
    print("="*50)
    
    # Dataset overview
    print(f"📊 Dataset Overview:")
    print(f"   • Total resumes analyzed: {len(df)}")
    print(f"   • Number of job categories: {df['Category'].nunique()}")
    print(f"   • Total skill mentions found: {len(skill_df)}")
    print(f"   • Unique skills identified: {skill_df['skill'].nunique()}")
    
    # Top categories
    print(f"\n🏆 Top Job Categories:")
    top_cats = df['Category'].value_counts().head(5)
    for cat, count in top_cats.items():
        pct = count/len(df)*100
        print(f"   • {cat}: {count} resumes ({pct:.1f}%)")
    
    # Skill insights
    print(f"\n🎯 Top Skills Overall:")
    top_skills = skill_df['skill'].value_counts().head(10)
    for skill, count in top_skills.items():
        pct = count/len(skill_df)*100
        print(f"   • {skill}: {count} mentions ({pct:.1f}%)")
    
    # Skill type distribution
    print(f"\n📈 Skill Type Distribution:")
    skill_types = skill_df['skill_type'].value_counts()
    for skill_type, count in skill_types.items():
        pct = count/len(skill_df)*100
        print(f"   • {skill_type.title()}: {count} ({pct:.1f}%)")
    
    # Category-specific insights
    print(f"\n🔍 Category-Specific Insights:")
    for category in top_cats.index[:3]:
        cat_skills = skill_df[skill_df['category'] == category]
        if not cat_skills.empty:
            top_cat_skill = cat_skills['skill'].value_counts().index[0]
            avg_skills = len(cat_skills) / df[df['Category'] == category].shape[0]
            print(f"   • {category}: Most common skill is '{top_cat_skill}', average {avg_skills:.1f} skills per resume")
    
    print("\n💡 RECOMMENDATIONS:")
    print("   • Focus on developing technical skills, especially Python and SQL")
    print("   • Don't neglect soft skills like communication and leadership")
    print("   • Tailor skill development based on target industry")
    print("   • Consider emerging technologies like cloud platforms (AWS, Azure)")
    
    print("\n✅ Analysis Complete!")
    print("   Generated visualizations and insights for skill demand analysis.")